In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import matplotlib.pyplot as plt

import operator
from functools import reduce
from timeit import default_timer

import sys
sys.path.append("..")
from mcno.utils import LpLoss

torch.manual_seed(0)
torch.cuda.manual_seed(0)  
torch.cuda.manual_seed_all(0) 

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Running on GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
else:
    print("Running on CPU")

In [ ]:
class AdaptiveMCConv1d(nn.Module):
    def __init__(self, in_channels, out_channels,  num_samples, grid_size):
        super(AdaptiveMCConv1d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.grid_size = grid_size
        self.num_samples = num_samples

        self.coeff_nn = nn.Parameter(torch.empty(in_channels, out_channels, num_samples))
        nn.init.xavier_uniform_(self.coeff_nn)
        with torch.no_grad():
            self.coeff_nn.copy_(F.normalize(self.coeff_nn, p=2, dim=-1))
        self.register_buffer('indices', torch.randint(0, self.grid_size, (self.num_samples,)))
        
    def forward(self, x):
        batch_size, _, _ = x.shape
        adaptive_samples = self.indices

        adaptive_samples_expanded = adaptive_samples.unsqueeze(0).unsqueeze(0).expand(batch_size, self.in_channels, self.num_samples)
        x_at_samples = torch.gather(x, dim=-1, index=adaptive_samples_expanded) 
        x_at_samples2 = x_at_samples.permute(0, 2, 1) 
        
        kernel_values = torch.einsum('bni,ion->bno', x_at_samples2, self.coeff_nn) 
        adaptive_contributions = torch.bmm(x_at_samples, kernel_values)  

        adaptive_contributions_resampled = F.interpolate(
            adaptive_contributions, 
            size= self.grid_size,             
            mode='linear',
            align_corners=False)
        
        return adaptive_contributions_resampled / self.num_samples

class SimpleBlock1d(nn.Module):
    def __init__(self, width,  grid_size, num_samples):
        super(SimpleBlock1d, self).__init__()

        self.width = width
        self.fc0 = nn.Linear(2, self.width)

        self.conv0 = AdaptiveMCConv1d(in_channels=width, out_channels=width, num_samples=num_samples, grid_size=grid_size)
        self.conv1 = AdaptiveMCConv1d(in_channels=width, out_channels=width, num_samples=num_samples, grid_size=grid_size)
        self.conv2 = AdaptiveMCConv1d(in_channels=width, out_channels=width, num_samples=num_samples, grid_size=grid_size)
        self.conv3 = AdaptiveMCConv1d(in_channels=width, out_channels=width, num_samples=num_samples, grid_size=grid_size)

        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        batchsize = x.shape[0]
        size_x = x.shape[1]

        x = self.fc0(x)
        x = x.permute(0, 2, 1)  

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = F.relu(x1 + x2)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = F.relu(x1 + x2)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = F.relu(x1 + x2)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = F.relu(x1 + x2)

        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

class Net1d(nn.Module):
    def __init__(self, width, grid_size, num_samples):
        super(Net1d, self).__init__()
        self.conv1 = SimpleBlock1d(width, grid_size, num_samples)

    def forward(self, x):
        x = self.conv1(x)
        return x.squeeze()

    def count_params(self):
        c = 0
        for p in self.parameters():
            c += reduce(operator.mul, list(p.size()))
        return c

# Configurations

In [ ]:
################################################################
#  configurations
################################################################
ntrain = 1000
ntest = 100

sub = 2**5 
h = 2**13 // sub 
s = h

batch_size = 20
learning_rate = 0.001
epochs =  500
step_size = 100
gamma = 0.5
width = 64

In [ ]:
from scipy.io import loadmat, savemat

rw_ = loadmat('../data/kdv_train_test.mat')
x_data = rw_['input'].astype(np.float32)
y_data = rw_['output'].astype(np.float32)

x_train = x_data[:ntrain,::sub]
y_train = y_data[:ntrain,::sub]
x_test = x_data[-ntest:,::sub]
y_test = y_data[-ntest:,::sub]

x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

grid = np.linspace(0, 1, s).reshape(1, s, 1)  
grid = torch.tensor(grid, dtype=torch.float)


x_train = torch.cat([x_train.reshape(ntrain, s, 1), grid.repeat(ntrain, 1, 1)], dim=2)
x_test = torch.cat([x_test.reshape(ntest, s, 1), grid.repeat(ntest, 1, 1)], dim=2)


train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False
)

In [ ]:
####################################
# Model Paramters
####################################

number_of_samples = 150 

model = Net1d(width, s, number_of_samples).to(device)
print(model.count_params())

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

In [ ]:
# Training loop
myloss = LpLoss(size_average=False)
for ep in range(epochs):
    model.train()
    t1 = default_timer()
    train_mse = 0
    train_l2 = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        
        mse = F.mse_loss(out, y, reduction='mean')
        l2 = myloss(out.view(batch_size, -1), y.view(batch_size, -1))
        l2.backward()

        optimizer.step()
        train_mse += mse.item()
        train_l2 += l2.item()

    scheduler.step()
    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            test_l2 += myloss(out.view(batch_size, -1), y.view(batch_size, -1)).item()

    train_mse /= len(train_loader)
    train_l2 /= ntrain
    test_l2 /= ntest

    t2 = default_timer()
    print(ep, t2-t1, train_mse, train_l2, test_l2)

In [ ]:
pred = torch.zeros(y_test.shape)
index = 0
results=[]
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=1, shuffle=False)
with torch.no_grad():
    for x, y in test_loader:
        test_l2 = 0
        x, y = x.to(device), y.to(device)
        current_batch_size = x.size(0) 

        out = model(x)
        pred[index] = out.cpu()

        test_l2 += myloss(out.view(1, -1), y.view(1, -1)).item()
        results.append(test_l2)
        print(index, test_l2)
        index = index + 1